# EDA — EEG Mind Reading

Goals:
1. Confirm data format, channel names, sampling rate
2. Check SNR: can we see evoked responses per video?
3. Visualize artifacts and decide on preprocessing thresholds

⚠️ Adjust the file paths to the actual data layout once released.

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from src.load_data import load_subject, load_all
from src.preprocess import bandpass, reject_artifacts, log_bandpower, per_video_average

In [ ]:
# point at one subject file once the data exists
ds = load_subject('../data/train/subject_01.npz')
print('X shape:', ds.X.shape)
print('labels:', np.unique(ds.y))
print('sfreq:', ds.sfreq)
print('channels:', ds.ch_names)

In [ ]:
# artifact overview
keep = reject_artifacts(ds.X, threshold_uv=150)
print(f'trials kept: {keep.sum()} / {len(keep)}')
for lab in np.unique(ds.y):
    m = ds.y == lab
    print(f'video {lab}: {keep[m].sum()}/{m.sum()} clean')

In [ ]:
Xf = bandpass(ds.X, ds.sfreq)
X_avg, labels = per_video_average(Xf, ds.y)
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
for ax, lab in zip(axes.ravel(), labels[:4]):
    ax.plot(ds.times, X_avg[lab].T * 1e6, lw=0.7)
    ax.set_title(f'video {lab - 1 if lab > 0 else 0}')
    ax.set_ylabel('µV')
plt.xlabel('Time (ms)')
plt.suptitle('Evoked (per-video average) responses')
plt.tight_layout()

In [ ]:
feat = log_bandpower(Xf, ds.sfreq)
print('bandpower features per trial:', feat.shape)
# separability preview: distance between video centroids
cents = np.stack([feat[ds.y == lab].reshape(len(ds.y[ds.y==lab]), -1).mean(0) for lab in np.unique(ds.y)])
d = np.linalg.norm(cents[:, None] - cents[None, :], axis=-1)
plt.imshow(d); plt.colorbar(); plt.title('Centroid distances in feature space')